# DistilBERT Toxicity Classifier — Google Colab Training Notebook

## Instructions
1. Upload your `train.csv` file when prompted (from the Jigsaw dataset)
2. Go to Runtime > Change Runtime Type > Select **T4 GPU**
3. Then go to Runtime > Run All
4. When training completes, all outputs will be zipped and
   downloaded automatically to your local machine
5. Extract the zip and copy contents into your project's
   `outputs/` folder

## What this notebook produces
- `outputs/model/` — saved DistilBERT model weights + tokenizer
- `outputs/figures/confusion_transformer.png`
- `outputs/figures/roc_curves.png` (both models on same plot)
- `outputs/transformer_metrics.json` — metrics to paste into paper.md
- `outputs/misclassifications_transformer.csv`

## Changes in this version
- ✅ FIX: `evaluation_strategy` → `eval_strategy`
- ✅ FIX: `WeightedTrainer` added to handle 90/10 class imbalance
- ✅ FIX: Threshold tuning at 0.35 instead of default 0.5
- ✅ FIX: `f1_macro...` key typo corrected to `f1_macro`

In [ ]:
# Verify GPU is available — if this shows CPU, stop and change
# Runtime > Change Runtime Type > GPU (T4)

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if device == "cpu":
    print("\n⚠️ WARNING: No GPU detected.")
    print("Training will take ~45-60 mins on CPU.")
    print("Recommended: Runtime > Change Runtime Type > T4 GPU")
else:
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install required packages — pinned versions for reproducibility
# This cell takes ~1-2 minutes

%%capture
!pip install transformers==4.38.2 \
             datasets==2.18.0 \
             accelerate==0.27.2 \
             scikit-learn==1.4.0 \
             seaborn==0.13.2 \
             matplotlib==3.8.2 \
             wordcloud==1.9.3 \
             tqdm==4.66.2

print("✅ All packages installed")

In [ ]:
# Standard library imports and basic logging configuration
import os
import json
import random
import logging
import zipfile
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Machine learning metrics and splitting
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, confusion_matrix,
    roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

# HuggingFace architecture and utilities
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import Dataset

# Colab file upload utility
from google.colab import files

# Configure logging to standard output
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s"
)
logger = logging.getLogger(__name__)

print("✅ All imports successful")

In [ ]:
# —— All hyperparameters in one place ——
# Change values here only — do not hardcode elsewhere in notebook

CONFIG = {
    # Data sampling and splitting
    "sample_n": 20000,
    "test_size": 0.2,
    "random_state": 42,
    "target_col": "toxic",
    "text_col": "comment_text",

    # Tokeniser settings
    "model_name": "distilbert-base-uncased",
    "max_length": 128,

    # Training — GPU settings (auto-adjusted for CPU below)
    "epochs": 3,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,

    # Decision threshold for imbalanced classification
    "threshold": 0.35,

    # Output directories
    "output_dir": "/content/outputs",
    "model_dir": "/content/outputs/model",
    "figures_dir": "/content/outputs/figures",
}

# Auto-adjust for CPU if no GPU available
if device == "cpu":
    CONFIG["batch_size"] = 8
    CONFIG["epochs"] = 2
    CONFIG["max_length"] = 64
    logger.warning("CPU mode: reduced batch_size=8, epochs=2, max_length=64")

# Set all random seeds for reproducibility
def set_all_seeds(seed: int) -> None:
    """Set seeds for Python, numpy, torch, and transformers."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

set_all_seeds(CONFIG["random_state"])

# Create output directories on local disk
for d in [CONFIG["output_dir"], CONFIG["model_dir"], CONFIG["figures_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ Configuration set")
print(f"   Sample size: {CONFIG['sample_n']:,}")
print(f"   Epochs: {CONFIG['epochs']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Max length: {CONFIG['max_length']}")
print(f"   Decision threshold: {CONFIG['threshold']}")
print(f"   Device: {device}")

In [ ]:
# Upload train.csv from your local machine
# A file picker will appear below — select your train.csv

print("📂 Please upload your train.csv file...")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df_full = pd.read_csv(filename)

# Validate required columns exist
required_cols = [CONFIG["text_col"], CONFIG["target_col"]]
missing = [c for c in required_cols if c not in df_full.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Perform a stratified sample to preserve class proportions
df, _ = train_test_split(
    df_full,
    train_size=CONFIG["sample_n"],
    stratify=df_full[CONFIG["target_col"]],
    random_state=CONFIG["random_state"]
)

toxic_count = df[CONFIG["target_col"]].sum()
nontoxic_count = len(df) - toxic_count
print(f"\n✅ Data loaded successfully")
print(f"   Total rows sampled: {len(df):,}")
print(f"   Toxic:     {toxic_count:,} ({toxic_count/len(df)*100:.1f}%)")
print(f"   Non-toxic: {nontoxic_count:,} ({nontoxic_count/len(df)*100:.1f}%)")

In [ ]:
import re

def clean_text(text: str) -> str:
    """
    Clean a single comment string.
    Steps: lowercase, remove HTML tags, remove URLs, collapse whitespace.
    Does NOT remove punctuation — preserves toxicity signals.
    """
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)          # strip HTML tags
    text = re.sub(r"http\S+|www\S+", " ", text)  # remove URLs
    text = re.sub(r"\s+", " ", text).strip()      # collapse whitespace
    return text

# Apply cleaning over the dataframe column
df["clean_text"] = df[CONFIG["text_col"]].apply(clean_text)

# Train / test split — stratified on toxic label
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"].tolist(),
    df[CONFIG["target_col"]].tolist(),
    test_size=CONFIG["test_size"],
    stratify=df[CONFIG["target_col"]],
    random_state=CONFIG["random_state"]
)

print(f"✅ Text cleaned and split")
print(f"   Train: {len(X_train):,} | Test: {len(X_test):,}")

In [ ]:
# Load DistilBERT tokeniser
tokenizer = DistilBertTokenizerFast.from_pretrained(CONFIG["model_name"])

def tokenize_batch(texts: list, labels: list) -> Dataset:
    """
    Tokenise a list of texts and return a HuggingFace Dataset.
    Returns Dataset with input_ids, attention_mask, and labels.
    """
    encoding = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=CONFIG["max_length"],
        return_tensors=None
    )
    encoding["labels"] = labels
    return Dataset.from_dict(encoding)

print("Tokenising training set...")
train_dataset = tokenize_batch(X_train, y_train)

print("Tokenising test set...")
test_dataset = tokenize_batch(X_test, y_test)

train_dataset.set_format("torch")
test_dataset.set_format("torch")

print(f"✅ Tokenisation complete")
print(f"   Train dataset: {len(train_dataset):,} examples")
print(f"   Test dataset:  {len(test_dataset):,} examples")

In [ ]:
# Load pretrained DistilBERT with a binary classification head
model = DistilBertForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=2
)
model.to(device)

print(f"✅ Model loaded: {CONFIG['model_name']}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ——————————————————————————————————————————————
# FIX 1: Compute class weights to handle 90/10 imbalance
# ——————————————————————————————————————————————
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"\n⚖️  Class weights → Non-toxic: {class_weights[0]:.2f} | Toxic: {class_weights[1]:.2f}")

# FIX 2: Custom Trainer that applies weighted loss during training
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Override loss function with class-weighted CrossEntropyLoss."""
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Penalise toxic misclassifications more heavily
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Define compute_metrics for Trainer callback
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics_trainer(eval_pred):
    """Compute accuracy and F1 at end of each epoch."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

# FIX 3: eval_strategy replaces deprecated evaluation_strategy
training_args = TrainingArguments(
    output_dir=CONFIG["model_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"] * 2,
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    eval_strategy="epoch",          # ✅ FIXED: was evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
    report_to="none",               # disable wandb
    seed=CONFIG["random_state"],
    no_cuda=(device == "cpu"),      # device is a plain string here
)

# Use WeightedTrainer instead of standard Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics_trainer,
)

print("\n🚀 Starting DistilBERT fine-tuning...")
print(f"   Epochs: {CONFIG['epochs']} | Batch: {CONFIG['batch_size']} | LR: {CONFIG['learning_rate']}")
print("   Progress will appear below. This takes ~10 mins on GPU.\n")

import time
start_time = time.time()
trainer.train()
elapsed = (time.time() - start_time) / 60
print(f"\n✅ Training complete in {elapsed:.1f} minutes")

trainer.save_model(CONFIG["model_dir"])
tokenizer.save_pretrained(CONFIG["model_dir"])
print(f"✅ Model saved to {CONFIG['model_dir']}")

In [ ]:
# Run inference on held-out test set
print("Running inference on test set...")

predictions = trainer.predict(test_dataset)
logits = predictions.predictions

# Convert logits to probabilities for the positive (toxic) class
y_proba = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
y_true = np.array(y_test)

# ——————————————————————————————————————————————
# FIX 4: Use tuned threshold (0.35) instead of default 0.5
# Lower threshold catches more toxic comments — helps with imbalance
# ——————————————————————————————————————————————
THRESHOLD = CONFIG["threshold"]
y_pred_default = np.argmax(logits, axis=1)
y_pred = (y_proba >= THRESHOLD).astype(int)

print(f"\n📊 Threshold comparison (test set):")
print(f"   Actual toxic comments:          {y_true.sum()}")
print(f"   Predicted toxic (threshold=0.5): {y_pred_default.sum()}")
print(f"   Predicted toxic (threshold={THRESHOLD}): {y_pred.sum()}")

# Compute all evaluation metrics using tuned threshold predictions
metrics = {
    "model": "DistilBERT (fine-tuned)",
    "threshold": THRESHOLD,
    "accuracy": round(accuracy_score(y_true, y_pred), 4),
    "f1_macro": round(f1_score(y_true, y_pred, average="macro"), 4),
    "f1_weighted": round(f1_score(y_true, y_pred, average="weighted"), 4),
    "precision_macro": round(precision_score(y_true, y_pred, average="macro"), 4),
    "recall_macro": round(recall_score(y_true, y_pred, average="macro"), 4),
    "roc_auc": round(roc_auc_score(y_true, y_proba), 4),
}

# Save metrics to JSON
metrics_path = f"{CONFIG['output_dir']}/transformer_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("\n" + "="*50)
print("DISTILBERT RESULTS")
print("="*50)
for k, v in metrics.items():
    print(f"  {k:<20} {v}")
print("="*50)
print(f"\n✅ Metrics saved to {metrics_path}")
print("\n📋 Copy these into your paper.md results table!")

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name, save_path):
    """Plot and save a normalised confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    cm_raw = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        cm, annot=False, fmt=".2f", cmap="Blues",
        xticklabels=["Non-Toxic", "Toxic"],
        yticklabels=["Non-Toxic", "Toxic"],
        ax=ax
    )
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.5,
                    f"{cm[i,j]:.2f}\n(n={cm_raw[i,j]:,})",
                    ha="center", va="center", fontsize=11,
                    color="white" if cm[i,j] > 0.5 else "black")

    ax.set_title(f"Confusion Matrix — {model_name}", fontsize=13, pad=12)
    ax.set_ylabel("True Label", fontsize=11)
    ax.set_xlabel("Predicted Label", fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {save_path}")

plot_confusion_matrix(
    y_true, y_pred,
    "DistilBERT (fine-tuned)",
    f"{CONFIG['figures_dir']}/confusion_transformer.png"
)

In [ ]:
# —— Baseline metrics (confirmed from your earlier run) ——
BASELINE_METRICS = {
    "accuracy": 0.940,
    "f1_macro": 0.833,
    "f1_weighted": 0.941,
    "roc_auc": 0.955,
}

# Re-fit baseline on same train/test split for fair ROC comparison
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

print("Re-fitting baseline on same train/test split for fair ROC comparison...")

baseline_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=3
    )),
    ("lr", LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=CONFIG["random_state"],
        class_weight="balanced"
    ))
])

baseline_pipeline.fit(X_train, y_train)
baseline_proba = baseline_pipeline.predict_proba(X_test)[:, 1]
print("✅ Baseline re-fitted")

def plot_roc_comparison(y_true, proba_baseline, proba_transformer, save_path):
    """Plot ROC curves for baseline and DistilBERT on the same axes."""
    fpr_b, tpr_b, _ = roc_curve(y_true, proba_baseline)
    fpr_t, tpr_t, _ = roc_curve(y_true, proba_transformer)
    auc_b = roc_auc_score(y_true, proba_baseline)
    auc_t = roc_auc_score(y_true, proba_transformer)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr_b, tpr_b, lw=2, color="#E07B39",
            label=f"TF-IDF + Logistic Regression (AUC = {auc_b:.3f})")
    ax.plot(fpr_t, tpr_t, lw=2, color="#3A7EBF",
            label=f"DistilBERT fine-tuned (AUC = {auc_t:.3f})")
    ax.plot([0,1], [0,1], "k--", lw=1, label="No-skill baseline")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("ROC Curve Comparison: Baseline vs DistilBERT", fontsize=13, pad=12)
    ax.legend(loc="lower right", fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {save_path}")

plot_roc_comparison(
    y_true, baseline_proba, y_proba,
    f"{CONFIG['figures_dir']}/roc_curves.png"
)

In [ ]:
def show_misclassifications(texts, y_true, y_pred, save_path, n=5):
    """
    Extract and save false positives and false negatives.
    Adds a commentary column explaining likely failure mode.
    """
    results = []
    fp_count = fn_count = 0

    for text, true, pred in zip(texts, y_true, y_pred):
        if pred == 1 and true == 0 and fp_count < n:
            results.append({
                "comment_text": text,
                "true_label": true,
                "predicted_label": pred,
                "error_type": "False Positive",
                "commentary": (
                    "Model flagged as toxic but comment is benign. "
                    "Likely triggered by surface-level keywords without "
                    "toxic intent (e.g. discussing violence academically, "
                    "reclaimed language, or dark humour)."
                )
            })
            fp_count += 1
        elif pred == 0 and true == 1 and fn_count < n:
            results.append({
                "comment_text": text,
                "true_label": true,
                "predicted_label": pred,
                "error_type": "False Negative",
                "commentary": (
                    "Model missed genuine toxicity. Likely caused by "
                    "subtle insults, sarcasm, coded language, or toxicity "
                    "expressed without common trigger terms."
                )
            })
            fn_count += 1
        if fp_count >= n and fn_count >= n:
            break

    df_errors = pd.DataFrame(results)
    df_errors.to_csv(save_path, index=False)

    print(f"\n📋 MISCLASSIFICATION EXAMPLES — DistilBERT")
    print("="*60)
    for _, row in df_errors.iterrows():
        print(f"\n[{row['error_type']}]")
        print(f"Text: {row['comment_text'][:120]}...")
        print(f"Commentary: {row['commentary']}")
    print(f"\n✅ Saved {len(df_errors)} misclassifications to {save_path}")

show_misclassifications(
    X_test, y_true, y_pred,
    f"{CONFIG['output_dir']}/misclassifications_transformer.csv"
)

In [ ]:
# Print side-by-side comparison of both models
comparison = pd.DataFrame([
    {
        "Model": "TF-IDF + Logistic Regression",
        "Accuracy": BASELINE_METRICS["accuracy"],
        "F1 (Macro)": BASELINE_METRICS["f1_macro"],
        "F1 (Weighted)": BASELINE_METRICS["f1_weighted"],
        "ROC-AUC": BASELINE_METRICS["roc_auc"],
    },
    {
        "Model": "DistilBERT (fine-tuned)",
        "Accuracy": metrics["accuracy"],
        "F1 (Macro)": metrics["f1_macro"],
        "F1 (Weighted)": metrics["f1_weighted"],
        "ROC-AUC": metrics["roc_auc"],
    }
])

comparison.to_csv(f"{CONFIG['output_dir']}/model_comparison.csv", index=False)

print("\n" + "="*65)
print("FULL MODEL COMPARISON")
print("="*65)
print(comparison.to_string(index=False))
print("="*65)

print("\n🏆 Winners by metric:")
for col in ["Accuracy", "F1 (Macro)", "F1 (Weighted)", "ROC-AUC"]:
    winner_idx = comparison[col].idxmax()
    winner = comparison.loc[winner_idx, "Model"]
    value = comparison.loc[winner_idx, col]
    print(f"   {col:<18} → {winner} ({value})")

print(f"\n✅ Comparison table saved to outputs/model_comparison.csv")
print("\n📋 Copy the DistilBERT row into your paper.md results table!")

In [ ]:
# Zip outputs (excluding model weights) and download
zip_path = "/content/toxicity_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(CONFIG["output_dir"]):
        if "model" in root:  # skip model weights — download separately if needed
            continue
        for filename in filenames:
            file_path = os.path.join(root, filename)
            arcname = os.path.relpath(file_path, "/content")
            zf.write(file_path, arcname)

print("✅ Zipped outputs (excluding model weights)")
print("📥 Downloading now...")
files.download(zip_path)

In [ ]:
# Only run this if you want the full DistilBERT model weights locally
# (~250 MB zip). Needed to run inference locally after training.

print("Zipping model weights (~250 MB, may take 1-2 mins)...")

model_zip_path = "/content/distilbert_model_weights.zip"
with zipfile.ZipFile(model_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(CONFIG["model_dir"]):
        for filename in filenames:
            file_path = os.path.join(root, filename)
            arcname = os.path.relpath(file_path, "/content")
            zf.write(file_path, arcname)

print("📥 Downloading model weights...")
files.download(model_zip_path)

## ✅ Colab Training Complete — Next Steps

### Files downloaded to your machine:
| File | Where to put it in your project |
|------|----------------------------------|
| `outputs/figures/confusion_transformer.png` | `outputs/figures/` |
| `outputs/figures/roc_curves.png` | `outputs/figures/` |
| `outputs/transformer_metrics.json` | `outputs/` |
| `outputs/misclassifications_transformer.csv` | `outputs/` |
| `outputs/model_comparison.csv` | `outputs/` |

### Changes made in this version:
- ✅ `evaluation_strategy` → `eval_strategy` (transformers >= 4.38)
- ✅ `WeightedTrainer` handles 90/10 class imbalance via weighted CrossEntropyLoss
- ✅ Decision threshold lowered to 0.35 to improve toxic recall
- ✅ `f1_macro...` key typo fixed to `f1_macro`
- ✅ `threshold` added to CONFIG for easy tuning

### Update your paper.md:
1. Open `outputs/transformer_metrics.json`
2. Fill in the DistilBERT row in your results table
3. Mention class weighting and threshold tuning in your methodology section